# Calculate Postal Code Centroids

This notebook calculates centroids for postal code areas from shapefile data.

**Process:**
1. Read postal code shapefile (plz-5stellig.shp)
2. Calculate centroid (geometric center) for each postal code polygon
3. Extract postal code and location information
4. Save centroids to CSV file for later use in IDW-KNN interpolation


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from tqdm import tqdm


In [ ]:
# Configuration
SHAPEFILE_PATH = Path("/mnt/d/heatpump_data/postal_code_data/postal_code_map/plz-5stellig.shp")
OUTPUT_FILE = Path("/mnt/d/heatpump_data/postal_code_data/postal_code_centroids.csv")

print(f"Shapefile path: {SHAPEFILE_PATH}")
print(f"Shapefile exists: {SHAPEFILE_PATH.exists()}")
print(f"Output file: {OUTPUT_FILE}")


Shapefile path: /mnt/d/heatpump_data/postal_code_data/postal_code_map/plz-5stellig.shp
Shapefile exists: True
Output file: /mnt/d/heatpump_data/postal_code_data/postal_code_centroids.csv


In [ ]:
# Read shapefile
print("Reading shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)

print(f"Total postal codes: {len(gdf)}")
print(f"Columns: {gdf.columns.tolist()}")
print(f"\nFirst few rows:")
print(gdf.head())
print(f"\nCRS: {gdf.crs}")


Reading shapefile...
Total postal codes: 8170
Columns: ['plz', 'note', 'einwohner', 'qkm', 'geometry']

First few rows:
     plz                            note  einwohner        qkm  \
0  81248                  81248 MÃ¼nchen        121   1.984763   
1  60315  60315 Frankfurt am Main (FOUR)          0   0.017481   
2  24988                  24988 Oeversee       3350  36.491463   
3  93185         93185 Michelsneukirchen       1786  32.873844   
4  93489                93489 Schorndorf       2622  38.597260   

                                            geometry  
0  POLYGON ((11.39468 48.14729, 11.3949 48.1478, ...  
1  POLYGON ((8.67254 50.11264, 8.67259 50.11264, ...  
2  POLYGON ((9.36586 54.69994, 9.36683 54.70014, ...  
3  POLYGON ((12.47666 49.13598, 12.47702 49.13637...  
4  POLYGON ((12.54904 49.19318, 12.54953 49.19371...  

CRS: EPSG:4326


In [ ]:
# Calculate centroids
print("Calculating centroids...")
gdf['centroid'] = gdf.geometry.centroid
gdf['centroid_lon'] = gdf['centroid'].x
gdf['centroid_lat'] = gdf['centroid'].y

print(f"Centroids calculated for {len(gdf)} postal codes")
print(f"\nSample centroids:")
print(gdf[['plz', 'centroid_lon', 'centroid_lat']].head() if 'plz' in gdf.columns else gdf[['centroid_lon', 'centroid_lat']].head())


Calculating centroids...
Centroids calculated for 8170 postal codes

Sample centroids:
     plz  centroid_lon  centroid_lat
0  81248     11.403147     48.148273
1  60315      8.673922     50.112310
2  24988      9.429714     54.707718
3  93185     12.541324     49.121808
4  93489     12.596062     49.167521


/tmp/ipykernel_15974/2800204479.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['centroid'] = gdf.geometry.centroid


In [ ]:
# Prepare output DataFrame
# Extract postal code column (may be named differently)
postal_code_col = None
for col in ['plz', 'PLZ', 'postal_code', 'POSTAL_CODE', 'postcode']:
    if col in gdf.columns:
        postal_code_col = col
        break

if postal_code_col is None:
    print("Warning: Could not find postal code column. Available columns:", gdf.columns.tolist())
    # Use index as postal code identifier
    centroids_df = pd.DataFrame({
        'postal_code': gdf.index.astype(str),
        'longitude': gdf['centroid_lon'],
        'latitude': gdf['centroid_lat']
    })
else:
    centroids_df = pd.DataFrame({
        'postal_code': gdf[postal_code_col].astype(str),
        'longitude': gdf['centroid_lon'],
        'latitude': gdf['centroid_lat']
    })

# Add any other relevant columns
if 'note' in gdf.columns:
    centroids_df['note'] = gdf['note']
elif 'ort' in gdf.columns:
    centroids_df['city'] = gdf['ort']

print(f"\nOutput DataFrame shape: {centroids_df.shape}")
print(f"\nFirst few rows:")
print(centroids_df.head())
print(f"\nData types:")
print(centroids_df.dtypes)



Output DataFrame shape: (8170, 4)

First few rows:
  postal_code  longitude   latitude                            note
0       81248  11.403147  48.148273                  81248 MÃ¼nchen
1       60315   8.673922  50.112310  60315 Frankfurt am Main (FOUR)
2       24988   9.429714  54.707718                  24988 Oeversee
3       93185  12.541324  49.121808         93185 Michelsneukirchen
4       93489  12.596062  49.167521                93489 Schorndorf

Data types:
postal_code     object
longitude      float64
latitude       float64
note            object
dtype: object


In [ ]:
# Save to CSV
print(f"\nSaving centroids to: {OUTPUT_FILE}")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
centroids_df.to_csv(OUTPUT_FILE, index=False)

print(f"✓ Saved {len(centroids_df)} postal code centroids")
print(f"\nSummary statistics:")
print(f"Longitude range: {centroids_df['longitude'].min():.6f} to {centroids_df['longitude'].max():.6f}")
print(f"Latitude range: {centroids_df['latitude'].min():.6f} to {centroids_df['latitude'].max():.6f}")



Saving centroids to: /mnt/d/heatpump_data/postal_code_data/postal_code_centroids.csv
✓ Saved 8170 postal code centroids

Summary statistics:
Longitude range: 5.971716 to 14.982235
Latitude range: 47.368732 to 55.019700
